# SmishGuard — train + export (Colab)

Trains a small encoder to classify Philippine SMS as `ham / scam / impersonation / otp_phish / loan / raffle`, exports int8 ONNX for Transformers.js, and prints an honest evaluation.

**Runtime → Change runtime type → T4 GPU.**

Either set `REPO_URL` below to the public SmishGuard repo, or upload the `training/*.py` files into `/content/training`.

In [ ]:
!pip -q install "transformers>=4.44" datasets "optimum[onnxruntime]" onnxruntime scikit-learn accelerate

In [ ]:
import os

REPO_URL = ""  # optional: e.g. https://github.com/<you>/smishguard
SCRIPTS = ["prepare_data.py", "finetune.py", "export_onnx.py", "eval.py"]

def has_scripts(d):
    return all(os.path.isfile(os.path.join(d, s)) for s in SCRIPTS)

if REPO_URL:
    !git clone -q $REPO_URL /content/smishguard
    os.chdir("/content/smishguard/training")
elif os.path.isdir("/content/training") and has_scripts("/content/training"):
    os.chdir("/content/training")
elif has_scripts("/content"):
    os.chdir("/content")

print("cwd:", os.getcwd())
missing = [s for s in SCRIPTS if not os.path.isfile(s)]
if missing:
    print("MISSING:", missing, "-> upload these into the Files pane, or set REPO_URL.")
else:
    print("Ready:", SCRIPTS)

## 1. Prepare data

One command pulls the public PH sources (scottleechua CC-BY-4.0 + Henit007 HF) and adds the English SMS Spam Collection for volume. Add your own Kaggle CSV with `--csv path`.

In [ ]:
!python prepare_data.py --ph-preset --bootstrap --out data
# with your own data too:
# !python prepare_data.py --ph-preset --bootstrap --csv /content/ph_spam.csv --out data

## 2. Fine-tune

Start with MiniLM. Only switch to `jcblaise/roberta-tagalog-base` if held-out F1 < 0.95.

In [ ]:
!python finetune.py --model Xenova/all-MiniLM-L6-v2 --data data --out model --epochs 5

## 3. Export int8 ONNX

In [ ]:
!python export_onnx.py --model-dir model --out onnx_out
!find onnx_out -maxdepth 2 -type f | sort

## 4. Evaluate honestly (copy these numbers into the README)

In [ ]:
!python eval.py --model-dir model --data data/test.csv

## 5. Publish to Hugging Face (public)

Create a token with **write** access, then run the cell. Afterwards set `VITE_MODEL_ID=<user>/<repo>` in the app `.env`.

In [ ]:
from huggingface_hub import HfApi, login

login()  # paste a write token when prompted
REPO_ID = "<your-username>/smishguard-minilm"
api = HfApi()
api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(folder_path="onnx_out", repo_id=REPO_ID, repo_type="model")
print("Uploaded to https://huggingface.co/" + REPO_ID)